In [2]:
import pandas as pd

df = pd.read_csv(r"C:\Users\TRISTAR\OneDrive\Documents\RAG chatbot 1st\RAG-\pharmaly_products_full.csv")

print("Dataset shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Dataset shape:
(9148, 9)

Columns:
['product_name', 'brand', 'categories', 'original_price', 'current_price', 'short_description', 'full_description', 'image_url', 'product_url']

First 5 rows:
                                        product_name             brand  \
0                    Cheks Postbiotic Balance  Bt/45             Cheks   
1  Acm Novophane Complément Alimentaire Ongles/ch...  Laboratoires ACM   
2                   Granions Biotine Vitamine B8 46g          Granions   
3  Nature’S Bounty Hair Skin & Nails  Multivitami...   Nature's Bounty   
4  Nature’S Bounty Hair,skin & Nails  Multivitami...   Nature's Bounty   

  categories original_price current_price  \
0      Sante            NaN   DA 2.700,00   
1      Sante            NaN   DA 6.500,00   
2      Sante            NaN   DA 4.800,00   
3      Sante            NaN   DA 8.200,00   
4      Sante            NaN   DA 7.800,00   

                                   short_description  \
0  Des gummies postbiotiques conçu

In [3]:
print("\nData types:")
print(df.dtypes)


Data types:
product_name         str
brand                str
categories           str
original_price       str
current_price        str
short_description    str
full_description     str
image_url            str
product_url          str
dtype: object


In [4]:
def clean_price(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()
    value = value.replace("DA", "").strip()

    # Algerian format: 2.700,00 → 2700.00
    value = value.replace(".", "")
    value = value.replace(",", ".")

    try:
        return float(value)
    except ValueError:
        return pd.NA


df["original_price"] = df["original_price"].apply(clean_price)
df["current_price"] = df["current_price"].apply(clean_price)

# Convert to numeric columns
df["original_price"] = pd.to_numeric(df["original_price"], errors="coerce")
df["current_price"] = pd.to_numeric(df["current_price"], errors="coerce")

In [5]:
print(df[["original_price", "current_price"]].head())
print()
print(df[["original_price", "current_price"]].dtypes)

   original_price  current_price
0             NaN         2700.0
1             NaN         6500.0
2             NaN         4800.0
3             NaN         8200.0
4             NaN         7800.0

original_price    float64
current_price     float64
dtype: object


In [6]:
print("Missing original prices:", df["original_price"].isna().sum())
print("Missing current prices:", df["current_price"].isna().sum())

print("Negative original prices:",
      (df["original_price"] < 0).sum())

print("Negative current prices:",
      (df["current_price"] < 0).sum())

print("Zero current prices:",
      (df["current_price"] == 0).sum())

print("Current price > original price:",
      ((df["current_price"] > df["original_price"]) &
       df["original_price"].notna()).sum())

Missing original prices: 9014
Missing current prices: 1
Negative original prices: 0
Negative current prices: 0
Zero current prices: 22
Current price > original price: 0


In [7]:
zero_price_products = df[df["current_price"] == 0]

print("Number of zero-price products:", len(zero_price_products))

print(
    zero_price_products[
        ["product_name", "brand", "current_price", "product_url"]
    ].to_string(index=False)
)

Number of zero-price products: 22
                                             product_name               brand  current_price                                                                                       product_url
   Soskin Stick Solaire Trés Haute Protectin Spf50+   15g              Soskin            0.0          https://www.pharmaly-dz.com/produit/soskin-stick-solaire-tres-haute-protectin-spf50-15g/
                            Arvea Mini Créme Miracle 50ml               ARVEA            0.0                                https://www.pharmaly-dz.com/produit/arvea-mini-creme-miracle-50ml/
                                           Arnica Gel Ngl                 NaN            0.0                                               https://www.pharmaly-dz.com/produit/arnica-gel-ngl/
             Hourglass 1.8 Bisque Vanish Concealer    6ml Hourglass Cosmetics            0.0                    https://www.pharmaly-dz.com/produit/hourglass-1-8-bisque-vanish-concealer-6ml/
           

In [8]:
missing_current_price = df[df["current_price"].isna()]

print("Number of products with missing current price:",
      len(missing_current_price))

print(
    missing_current_price[
        ["product_name", "brand", "current_price", "product_url"]
    ].to_string(index=False)
)

Number of products with missing current price: 1
         product_name    brand  current_price                                                product_url
Sensibio Gel Moussant Bioderma            NaN https://www.pharmaly-dz.com/produit/sensibio-gel-moussant/


In [9]:
df.loc[df["current_price"] == 0, "current_price"] = pd.NA

In [10]:
print("Missing current prices:", df["current_price"].isna().sum())
print("Zero current prices:", (df["current_price"] == 0).sum())

Missing current prices: 23
Zero current prices: 0


In [11]:
print(df.isnull().sum())

product_name            0
brand                  96
categories              0
original_price       9014
current_price          23
short_description    1704
full_description        0
image_url               0
product_url             0
dtype: int64


In [12]:
print("\nDuplicate product names:")
print(df["product_name"].duplicated().sum())


Duplicate product names:
4


In [13]:
duplicate_names = df[df["product_name"].duplicated(keep=False)].sort_values("product_name")

print(
    duplicate_names[
        ["product_name", "brand", "categories", "current_price", "product_url"]
    ].to_string(index=False)
)

                                product_name               brand categories  current_price                                                                                     product_url
Hourglass 1.8 Bisque Vanish Concealer    6ml Hourglass Cosmetics Maquillage         7200.0                https://www.pharmaly-dz.com/produit/hourglass-1-8-bisque-vanish-concealer-6ml-2/
Hourglass 1.8 Bisque Vanish Concealer    6ml Hourglass Cosmetics Maquillage            NaN                  https://www.pharmaly-dz.com/produit/hourglass-1-8-bisque-vanish-concealer-6ml/
           La Roche-posay Cicaplast Baume B5      La Roche-Posay     Visage         2500.0 https://www.pharmaly-dz.com/produit/maybelline-instant-antiage-perfector-4-in-one-glow-00-20ml/
           La Roche-posay Cicaplast Baume B5      La Roche-Posay     Visage         4200.0                              https://www.pharmaly-dz.com/produit/lrp-cicaplaste-baume-b5-100ml/
            L’Oréal Magic Retouch Blond 75ml       L'Oréal Paris 

In [14]:
print("Duplicate product URLs:")
print(df["product_url"].duplicated().sum())

Duplicate product URLs:
0


In [15]:
print("Missing product URLs:", df["product_url"].isna().sum())

print(
    "URLs not starting with http:",
    (~df["product_url"].str.startswith("http", na=False)).sum()
)

print(
    "Unique product URLs:",
    df["product_url"].nunique()
)

Missing product URLs: 0
URLs not starting with http: 0
Unique product URLs: 9148


In [16]:
df["brand"] = df["brand"].fillna("Not specified")

In [17]:
print("Missing brands:", df["brand"].isna().sum())
print(df["brand"].value_counts().head(10))

Missing brands: 0
brand
L'Oréal Paris              598
NYX Professional Makeup    368
Maybelline New York        313
Garnier                    220
Flux Care                  180
Nivea                      173
SHEGLAM                    141
Medicube                   126
KIKO Milano                125
La Roche-Posay             119
Name: count, dtype: int64


In [18]:
print("\nMissing product URLs:")
print(df["product_url"].isna().sum())

print("\nUnique product URLs:")
print(df["product_url"].nunique())


Missing product URLs:
0

Unique product URLs:
9148


Cleaned structure: We are keeping the columns: product name, brand, categories, original price, current price, short description, full description, and product URL. The image URL will not be used for the RAG chatbot. The product URL will be preserved because it is required for source citations. Each product will later be represented as one document with its metadata.

In [19]:
# Clean spaces in text columns

text_columns = [
    "product_name",
    "brand",
    "categories",
    "short_description",
    "full_description",
    "product_url"
]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

In [20]:
print(df[[
    "product_name",
    "brand",
    "categories"
]].head(10))

                                        product_name             brand  \
0                    Cheks Postbiotic Balance  Bt/45             Cheks   
1  Acm Novophane Complément Alimentaire Ongles/ch...  Laboratoires ACM   
2                   Granions Biotine Vitamine B8 46g          Granions   
3  Nature’S Bounty Hair Skin & Nails  Multivitami...   Nature's Bounty   
4  Nature’S Bounty Hair,skin & Nails  Multivitami...   Nature's Bounty   
5               Nature’S Bounty Hair Growth   90 Cap   Nature's Bounty   
6                           Cheks Memory & Focus 30u             Cheks   
7                           Cheks Memory & Focus 45u             Cheks   
8                      Cheks Kids Magnesium Calm 30u             Cheks   
9                             Cheks Kids Iron + 30 U             Cheks   

  categories  
0      Sante  
1      Sante  
2      Sante  
3      Sante  
4      Sante  
5      Sante  
6      Sante  
7      Sante  
8      Sante  
9      Sante  


In [21]:
print("\nMissing values after text cleaning:")
print(df.isnull().sum())


Missing values after text cleaning:
product_name            0
brand                   0
categories              0
original_price       9014
current_price          23
short_description    1704
full_description        0
image_url               0
product_url             0
dtype: int64


In [22]:
import re

def clean_text(text):
    if pd.isna(text):
        return pd.NA

    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

for col in text_columns:
    df[col] = df[col].apply(clean_text)

In [23]:
no_description = df[
    df["short_description"].isna() &
    df["full_description"].isna()
]

print("Number of products without any description:")
print(len(no_description))

print("\nProducts without descriptions:")
print(no_description[
    ["product_name", "brand", "categories", "current_price", "product_url"]
].to_string(index=False))

Number of products without any description:
0

Products without descriptions:
Empty DataFrame
Columns: [product_name, brand, categories, current_price, product_url]
Index: []


In [24]:
df["description"] = df["full_description"]

df["description"] = df["description"].fillna(
    df["short_description"]
)

In [25]:
df["description"] = df["description"].apply(clean_text)

In [26]:
print("Missing descriptions:", df["description"].isna().sum())

Missing descriptions: 0


In [27]:
print("Unknown values in each column:")

for col in df.columns:
    count = (df[col].astype("string").str.strip().str.lower() == "unknown").sum()
    if count > 0:
        print(f"{col}: {count}")

Unknown values in each column:


In [28]:
# Convert "Unknown" brand values to missing values

df["brand"] = df["brand"].replace("Unknown", pd.NA)

print("Missing brands:")
print(df["brand"].isnull().sum())

Missing brands:
0


In [29]:
print(df[["original_price", "current_price"]].describe())

       original_price  current_price
count      134.000000    9125.000000
mean      4643.283582    2909.636221
std       2179.792629    2655.465446
min        900.000000     100.000000
25%       3800.000000    1100.000000
50%       4500.000000    2000.000000
75%       5000.000000    4000.000000
max      14000.000000   42500.000000


In [30]:
print("\nMissing prices:")
print(df[["original_price", "current_price"]].isnull().sum())


Missing prices:
original_price    9014
current_price       23
dtype: int64


In [31]:
print("Negative current prices:")
print((df["current_price"] < 0).sum())

print("\nZero current prices:")
print((df["current_price"] == 0).sum())

print("\nNegative original prices:")
print((df["original_price"] < 0).sum())

Negative current prices:
0

Zero current prices:
0

Negative original prices:
0


In [32]:
price_problem = df[
    df["original_price"].notna() &
    (df["current_price"] > df["original_price"])
]

print("Products where current price > original price:")
print(len(price_problem))

print(
    price_problem[
        ["product_name", "original_price", "current_price"]
    ].to_string(index=False)
)

Products where current price > original price:
0
Empty DataFrame
Columns: [product_name, original_price, current_price]
Index: []


In [33]:
duplicate_names = df[
    df["product_name"].duplicated(keep=False)
].sort_values("product_name")

print(
    duplicate_names[
        ["product_name", "brand", "current_price", "product_url"]
    ].to_string(index=False)
)

                             product_name               brand  current_price                                                                                     product_url
Hourglass 1.8 Bisque Vanish Concealer 6ml Hourglass Cosmetics         7200.0                https://www.pharmaly-dz.com/produit/hourglass-1-8-bisque-vanish-concealer-6ml-2/
Hourglass 1.8 Bisque Vanish Concealer 6ml Hourglass Cosmetics            NaN                  https://www.pharmaly-dz.com/produit/hourglass-1-8-bisque-vanish-concealer-6ml/
        La Roche-posay Cicaplast Baume B5      La Roche-Posay         2500.0 https://www.pharmaly-dz.com/produit/maybelline-instant-antiage-perfector-4-in-one-glow-00-20ml/
        La Roche-posay Cicaplast Baume B5      La Roche-Posay         4200.0                              https://www.pharmaly-dz.com/produit/lrp-cicaplaste-baume-b5-100ml/
         L’Oréal Magic Retouch Blond 75ml       L'Oréal Paris         1000.0                          https://www.pharmaly-dz.com/produ

In [34]:
print("Duplicate product URLs:")
print(df["product_url"].duplicated().sum())

Duplicate product URLs:
0


In [35]:
duplicate_urls = df[
    df["product_url"].duplicated(keep=False)
].sort_values("product_url")

print(
    duplicate_urls[
        ["product_name", "brand", "current_price", "product_url"]
    ].to_string(index=False)
)

Empty DataFrame
Columns: [product_name, brand, current_price, product_url]
Index: []


In [36]:
print("Missing product URLs:")
print(df["product_url"].isnull().sum())

Missing product URLs:
0


In [37]:
print("URLs that don't start with http:")
print(
    (~df["product_url"].str.startswith("http", na=False)).sum()
)

URLs that don't start with http:
0


In [52]:
rag_columns = [
    "product_name",
    "brand",
    "categories",
    "original_price",
    "current_price",
    "description",
    "product_url"
]

df_rag = df[rag_columns].copy()

print("RAG dataset shape:", df_rag.shape)
print("\nRAG dataset columns:")
print(df_rag.columns.tolist())

display(df_rag.head())

RAG dataset shape: (9148, 7)

RAG dataset columns:
['product_name', 'brand', 'categories', 'original_price', 'current_price', 'description', 'product_url']


,product_name,brand,categories,original_price,current_price,description,product_url
0,Cheks Postbiotic Balance Bt/45,Cheks,Sante,NaN,2700.0,Le Cheks Postbiotic Balance est un complément ...,https://www.pharmaly-dz.com/produit/cheks-post...
1,Acm Novophane Complément Alimentaire Ongles/ch...,Laboratoires ACM,Sante,NaN,6500.0,La crème apaisante ACM Sédacalm 120ml est spéc...,https://www.pharmaly-dz.com/produit/acm-novoph...
2,Granions Biotine Vitamine B8 46g,Granions,Sante,NaN,4800.0,Granions Biotine Vitamine B8 est un complément...,https://www.pharmaly-dz.com/produit/granions-b...
3,Nature’S Bounty Hair Skin & Nails Multivitamin...,Nature's Bounty,Sante,NaN,8200.0,"Nature’s Bounty Hair, Skin & Nails Multivitami...",https://www.pharmaly-dz.com/produit/natures-bo...
4,"Nature’S Bounty Hair,skin & Nails Multivitamin...",Nature's Bounty,Sante,NaN,7800.0,"Nature’s Bounty Hair, Skin & Nails Multivitami...",https://www.pharmaly-dz.com/produit/natures-bo...


In [53]:
output_path = r"C:\Users\TRISTAR\Downloads\pharmaly_products_rag_ready.csv"

df_rag.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Saved successfully!")
print(output_path)

Saved successfully!
C:\Users\TRISTAR\Downloads\pharmaly_products_rag_ready.csv


In [54]:
documents = []
metadata = []

for _, row in df_rag.iterrows():

    document = f"""
Product Name: {row['product_name']}
Brand: {row['brand'] if pd.notna(row['brand']) else 'Not specified'}
Category: {row['categories']}

Original Price: {row['original_price'] if pd.notna(row['original_price']) else 'Not specified'}
Current Price: {row['current_price']}

Description:
{row['description'] if row['description'] else 'No description available'}
""".strip()

    documents.append(document)

    metadata.append({
        "product_name": row["product_name"],
        "brand": row["brand"] if pd.notna(row["brand"]) else "",
        "category": row["categories"],
        "original_price": (
            float(row["original_price"])
            if pd.notna(row["original_price"])
            else None
        ),
        "current_price": float(row["current_price"]),
        "product_url": row["product_url"]
    })

print("Number of documents:", len(documents))
print("Number of metadata:", len(metadata))

Number of documents: 9148
Number of metadata: 9148


In [55]:
print(documents[0])

Product Name: Cheks Postbiotic Balance Bt/45
Brand: Cheks
Category: Sante

Original Price: Not specified
Current Price: 2700.0

Description:
Le Cheks Postbiotic Balance est un complément alimentaire en gummies conçu pour soutenir l’équilibre du microbiote et le confort digestif. Sa formule associe un lysat postbiotique de Lactobacillus plantarum, du butyrate de calcium, de l’inuline ainsi que des extraits de gingembre et de menthe. Cette synergie accompagne la digestion, la régularité du transit et aide à limiter les sensations de ballonnement après les repas. Le Cheks Postbiotic Balance contient également du zinc et de la vitamine D3, qui contribuent au fonctionnement normal du système immunitaire. Ses gummies vegan sont sans sucre ajouté, sans gélatine et sans gluten.


In [56]:
print("\nMetadata:")
print(metadata[0])


Metadata:
{'product_name': 'Cheks Postbiotic Balance Bt/45', 'brand': 'Cheks', 'category': 'Sante', 'original_price': None, 'current_price': 2700.0, 'product_url': 'https://www.pharmaly-dz.com/produit/cheks-postbiotic-balance-bt-45/'}


In [57]:
print("Number of products:", len(df_rag))
print("Number of documents:", len(documents))
print("Number of metadata records:", len(metadata))

Number of products: 9148
Number of documents: 9148
Number of metadata records: 9148


In [58]:
for i in range(3):
    print("=" * 70)
    print(f"DOCUMENT {i + 1}")
    print("=" * 70)
    print(documents[i])
    print()
    print("METADATA:")
    print(metadata[i])
    print()

DOCUMENT 1
Product Name: Cheks Postbiotic Balance Bt/45
Brand: Cheks
Category: Sante

Original Price: Not specified
Current Price: 2700.0

Description:
Le Cheks Postbiotic Balance est un complément alimentaire en gummies conçu pour soutenir l’équilibre du microbiote et le confort digestif. Sa formule associe un lysat postbiotique de Lactobacillus plantarum, du butyrate de calcium, de l’inuline ainsi que des extraits de gingembre et de menthe. Cette synergie accompagne la digestion, la régularité du transit et aide à limiter les sensations de ballonnement après les repas. Le Cheks Postbiotic Balance contient également du zinc et de la vitamine D3, qui contribuent au fonctionnement normal du système immunitaire. Ses gummies vegan sont sans sucre ajouté, sans gélatine et sans gluten.

METADATA:
{'product_name': 'Cheks Postbiotic Balance Bt/45', 'brand': 'Cheks', 'category': 'Sante', 'original_price': None, 'current_price': 2700.0, 'product_url': 'https://www.pharmaly-dz.com/produit/cheks-

In [59]:
import json

output_json = r"C:\Users\TRISTAR\Downloads\pharmaly_documents.json"

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(
        {
            "documents": documents,
            "metadata": metadata
        },
        f,
        ensure_ascii=False,
        indent=2
    )

print("Documents saved successfully!")
print(output_json)

Documents saved successfully!
C:\Users\TRISTAR\Downloads\pharmaly_documents.json


In [60]:
with open(output_json, "r", encoding="utf-8") as f:
    saved_data = json.load(f)

print("Number of documents:", len(saved_data["documents"]))
print("Number of metadata records:", len(saved_data["metadata"]))

print("\nFirst document:")
print(saved_data["documents"][0])

print("\nFirst metadata:")
print(saved_data["metadata"][0])

Number of documents: 9148
Number of metadata records: 9148

First document:
Product Name: Cheks Postbiotic Balance Bt/45
Brand: Cheks
Category: Sante

Original Price: Not specified
Current Price: 2700.0

Description:
Le Cheks Postbiotic Balance est un complément alimentaire en gummies conçu pour soutenir l’équilibre du microbiote et le confort digestif. Sa formule associe un lysat postbiotique de Lactobacillus plantarum, du butyrate de calcium, de l’inuline ainsi que des extraits de gingembre et de menthe. Cette synergie accompagne la digestion, la régularité du transit et aide à limiter les sensations de ballonnement après les repas. Le Cheks Postbiotic Balance contient également du zinc et de la vitamine D3, qui contribuent au fonctionnement normal du système immunitaire. Ses gummies vegan sont sans sucre ajouté, sans gélatine et sans gluten.

First metadata:
{'product_name': 'Cheks Postbiotic Balance Bt/45', 'brand': 'Cheks', 'category': 'Sante', 'original_price': None, 'current_pri